In [1]:
# Step 1: Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Step 2: Load Dataset

users = pd.read_csv(
    "users.dat",
    sep="::",
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip_Code"],
    encoding="latin-1"
)

ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
    encoding="latin-1"
)

movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Movie_Title", "Genres"],
    encoding="latin-1"
)

print("Datasets loaded successfully.")

print("\nUsers dataset shape:", users.shape)
print("Ratings dataset shape:", ratings.shape)
print("Movies dataset shape:", movies.shape)

Datasets loaded successfully.

Users dataset shape: (6040, 5)
Ratings dataset shape: (1000209, 4)
Movies dataset shape: (3883, 3)


In [3]:
# Step 3: Understand Dataset

print("===== USERS DATASET =====")
print(users.head())
print("\nShape:", users.shape)
print("\nData Types:")
print(users.dtypes)
print("\nMissing Values:")
print(users.isnull().sum())

print("\n\n===== RATINGS DATASET =====")
print(ratings.head())
print("\nShape:", ratings.shape)
print("\nData Types:")
print(ratings.dtypes)
print("\nMissing Values:")
print(ratings.isnull().sum())

print("\n\n===== MOVIES DATASET =====")
print(movies.head())
print("\nShape:", movies.shape)
print("\nData Types:")
print(movies.dtypes)
print("\nMissing Values:")
print(movies.isnull().sum())

===== USERS DATASET =====
   UserID Gender  Age  Occupation Zip_Code
0       1      F    1          10    48067
1       2      M   56          16    70072
2       3      M   25          15    55117
3       4      M   45           7    02460
4       5      M   25          20    55455

Shape: (6040, 5)

Data Types:
UserID        int64
Gender          str
Age           int64
Occupation    int64
Zip_Code        str
dtype: object

Missing Values:
UserID        0
Gender        0
Age           0
Occupation    0
Zip_Code      0
dtype: int64


===== RATINGS DATASET =====
   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291

Shape: (1000209, 4)

Data Types:
UserID       int64
MovieID      int64
Rating       int64
Timestamp    int64
dtype: object

Missing Values:
UserID       0
MovieID      0
Rating       0
Timestamp    0
dt

In [4]:
print("===== DATASET SUMMARY =====")

print("Unique Users:", ratings["UserID"].nunique())
print("Unique Movies in Ratings:", ratings["MovieID"].nunique())
print("Unique Movies in Movie Data:", movies["MovieID"].nunique())

print("\nRating Range:", ratings["Rating"].min(), "to", ratings["Rating"].max())
print("Average Rating:", round(ratings["Rating"].mean(), 2))

print("\nRating Distribution:")
print(ratings["Rating"].value_counts().sort_index())

===== DATASET SUMMARY =====
Unique Users: 6040
Unique Movies in Ratings: 3706
Unique Movies in Movie Data: 3883

Rating Range: 1 to 5
Average Rating: 3.58

Rating Distribution:
Rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64


In [5]:
# Step 4: Data Cleaning

print("===== BEFORE CLEANING =====")
print("Ratings:", len(ratings))
print("Duplicate rating records:", ratings.duplicated().sum())

# Check for invalid ratings
print("Invalid ratings:", (~ratings["Rating"].isin([1, 2, 3, 4, 5])).sum())

# Remove exact duplicate records
ratings = ratings.drop_duplicates().copy()

print("\n===== AFTER CLEANING =====")
print("Ratings:", len(ratings))
print("Duplicate rating records:", ratings.duplicated().sum())

print("\nMissing values after cleaning:")
print(ratings.isnull().sum())

===== BEFORE CLEANING =====
Ratings: 1000209
Duplicate rating records: 0
Invalid ratings: 0

===== AFTER CLEANING =====
Ratings: 1000209
Duplicate rating records: 0

Missing values after cleaning:
UserID       0
MovieID      0
Rating       0
Timestamp    0
dtype: int64


In [7]:
# Step 5: Feature Selection

# Features required for Collaborative Filtering
cf_features = ['UserID', 'MovieID', 'Rating']

ratings_cf = ratings[cf_features].copy()

# Movie information for displaying recommendations
movie_features = movies[['MovieID', 'Movie_Title', 'Genres']].copy()

print("Collaborative Filtering Features:")
print(ratings_cf.columns.tolist())

print("\nRatings dataset shape:", ratings_cf.shape)

print("\nMovie Information Features:")
print(movie_features.columns.tolist())

print("\nMovie information shape:", movie_features.shape)

Collaborative Filtering Features:
['UserID', 'MovieID', 'Rating']

Ratings dataset shape: (1000209, 3)

Movie Information Features:
['MovieID', 'Movie_Title', 'Genres']

Movie information shape: (3883, 3)


In [8]:
# Step 6: Feature Preprocessing

# Create User-Movie Rating Matrix
user_movie_matrix = ratings_cf.pivot_table(
    index='UserID',
    columns='MovieID',
    values='Rating'
)

print("User-Movie Rating Matrix created successfully.")
print("Matrix shape:", user_movie_matrix.shape)
print("\nFirst 5 rows and 5 columns:")
print(user_movie_matrix.iloc[:5, :5])

User-Movie Rating Matrix created successfully.
Matrix shape: (6040, 3706)

First 5 rows and 5 columns:
MovieID    1   2   3   4   5
UserID                      
1        5.0 NaN NaN NaN NaN
2        NaN NaN NaN NaN NaN
3        NaN NaN NaN NaN NaN
4        NaN NaN NaN NaN NaN
5        NaN NaN NaN NaN NaN


In [9]:
# Step 7: Convert Features into Numerical Form

# Replace missing ratings with 0 for collaborative filtering calculations
user_movie_matrix_filled = user_movie_matrix.fillna(0)

print("Features converted into numerical form successfully.")

print("\nMatrix shape:", user_movie_matrix_filled.shape)
print("Matrix data type:", user_movie_matrix_filled.dtypes.iloc[0])

print("\nFirst 5 rows and 5 columns:")
print(user_movie_matrix_filled.iloc[:5, :5])

Features converted into numerical form successfully.

Matrix shape: (6040, 3706)
Matrix data type: float64

First 5 rows and 5 columns:
MovieID    1    2    3    4    5
UserID                          
1        5.0  0.0  0.0  0.0  0.0
2        0.0  0.0  0.0  0.0  0.0
3        0.0  0.0  0.0  0.0  0.0
4        0.0  0.0  0.0  0.0  0.0
5        0.0  0.0  0.0  0.0  0.0


In [18]:
# Step 8: Create Feature Matrix

feature_matrix = user_movie_matrix_filled.values

print("Feature matrix created successfully.")

print("Feature matrix shape:", feature_matrix.shape)
print("Feature matrix type:", type(feature_matrix))
print("Feature matrix data type:", feature_matrix.dtype)

print("\nFirst 5 rows and 5 columns:")
print(feature_matrix[:5, :5])

Feature matrix created successfully.
Feature matrix shape: (6040, 3706)
Feature matrix type: <class 'numpy.ndarray'>
Feature matrix data type: float64

First 5 rows and 5 columns:
[[5. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]


In [19]:
# Step 9: Calculate Cosine Similarity

# Calculate similarity between users
user_similarity = cosine_similarity(feature_matrix)

print("Cosine similarity calculated successfully.")

print("Similarity matrix shape:", user_similarity.shape)
print("Similarity matrix type:", type(user_similarity))
print("Similarity matrix data type:", user_similarity.dtype)

print("\nFirst 5 users similarity:")
print(user_similarity[:5, :5])

Cosine similarity calculated successfully.
Similarity matrix shape: (6040, 6040)
Similarity matrix type: <class 'numpy.ndarray'>
Similarity matrix data type: float64

First 5 users similarity:
[[1.         0.09638153 0.12060981 0.13245483 0.09015844]
 [0.09638153 1.         0.1514786  0.17117618 0.11439362]
 [0.12060981 0.1514786  1.         0.15122669 0.06290733]
 [0.13245483 0.17117618 0.15122669 1.         0.04509367]
 [0.09015844 0.11439362 0.06290733 0.04509367 1.        ]]


In [20]:
# Create Sparse User-Movie Matrix for Deployment

from scipy.sparse import csr_matrix

sparse_feature_matrix = csr_matrix(
    user_movie_matrix_filled.values
)

print("Sparse feature matrix created successfully.")
print("Shape:", sparse_feature_matrix.shape)
print("Non-zero values:", sparse_feature_matrix.nnz)
print("Sparse matrix type:", type(sparse_feature_matrix))

Sparse feature matrix created successfully.
Shape: (6040, 3706)
Non-zero values: 1000209
Sparse matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [21]:
# Create Lightweight Deployment Files

import pickle
import os

os.makedirs("model/deployment", exist_ok=True)

# Save sparse user-movie matrix
with open("model/deployment/sparse_feature_matrix.pkl", "wb") as file:
    pickle.dump(sparse_feature_matrix, file)

# Save User IDs
with open("model/deployment/user_ids.pkl", "wb") as file:
    pickle.dump(user_movie_matrix.index.to_numpy(), file)

# Save Movie IDs
with open("model/deployment/movie_ids.pkl", "wb") as file:
    pickle.dump(user_movie_matrix.columns.to_numpy(), file)

# Save user clusters
with open("model/deployment/user_clusters.pkl", "wb") as file:
    pickle.dump(user_cluster_data, file)

# Save movie information
with open("model/deployment/movie_features.pkl", "wb") as file:
    pickle.dump(movie_features, file)

print("Lightweight deployment files saved successfully.")
print("\nFiles:")
print(os.listdir("model/deployment"))

Lightweight deployment files saved successfully.

Files:
['movie_features.pkl', 'movie_ids.pkl', 'sparse_feature_matrix.pkl', 'user_clusters.pkl', 'user_ids.pkl']


In [12]:
# Step 10: Build Recommendation Function

def recommend_movies(user_id, num_recommendations=10, num_similar_users=20):
    
    # Get the index of the selected user
    user_index = user_movie_matrix.index.get_loc(user_id)
    
    # Get similarity scores for the selected user
    similarity_scores = user_similarity[user_index]
    
    # Get indices of most similar users (excluding the user themselves)
    similar_user_indices = np.argsort(similarity_scores)[::-1][1:num_similar_users + 1]
    
    # Get the movie ratings of similar users
    similar_users_ratings = feature_matrix[similar_user_indices]
    
    # Calculate weighted rating scores
    similar_scores = similarity_scores[similar_user_indices]
    
    weighted_scores = np.dot(similar_scores, similar_users_ratings)
    
    # Get movies already rated by the target user
    user_ratings = feature_matrix[user_index]
    
    # Do not recommend movies already rated by the user
    weighted_scores[user_ratings > 0] = -1
    
    # Get indices of movies with highest recommendation scores
    recommended_indices = np.argsort(weighted_scores)[::-1][:num_recommendations]
    
    # Get Movie IDs
    recommended_movie_ids = user_movie_matrix.columns[recommended_indices]
    
    # Create recommendation dataframe
    recommendations = movie_features[
        movie_features["MovieID"].isin(recommended_movie_ids)
    ].copy()
    
    # Preserve recommendation order
    recommendation_order = {
        movie_id: rank
        for rank, movie_id in enumerate(recommended_movie_ids)
    }
    
    recommendations["Recommendation_Rank"] = (
        recommendations["MovieID"].map(recommendation_order)
    )
    
    recommendations["Recommendation_Score"] = (
        recommendations["MovieID"]
        .map(dict(zip(recommended_movie_ids, weighted_scores[recommended_indices])))
    )
    
    recommendations = recommendations.sort_values("Recommendation_Rank")
    
    return recommendations.reset_index(drop=True)


print("Recommendation function created successfully.")

Recommendation function created successfully.


In [13]:
# Step 11: Test Recommendations

recommendations = recommend_movies(
    user_id=1,
    num_recommendations=10,
    num_similar_users=20
)

print("Recommended Movies for User 1:")
print(recommendations)

Recommended Movies for User 1:
   MovieID                                        Movie_Title  \
0     2081                         Little Mermaid, The (1989)   
1     2078                            Jungle Book, The (1967)   
2     2096                             Sleeping Beauty (1959)   
3      364                              Lion King, The (1994)   
4     1282                                    Fantasia (1940)   
5     2085                              101 Dalmatians (1961)   
6     2087                                   Peter Pan (1953)   
7     1210  Star Wars: Episode VI - Return of the Jedi (1983)   
8      480                               Jurassic Park (1993)   
9     2080                          Lady and the Tramp (1955)   

                                        Genres  Recommendation_Rank  \
0  Animation|Children's|Comedy|Musical|Romance                    0   
1          Animation|Children's|Comedy|Musical                    1   
2                 Animation|Children's|M

In [14]:
# K-Means User Clustering

from sklearn.cluster import KMeans

# Create user clusters
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

user_clusters = kmeans.fit_predict(feature_matrix)

# Store cluster information
user_cluster_data = pd.DataFrame({
    "UserID": user_movie_matrix.index,
    "Cluster": user_clusters
})

print("K-Means clustering completed successfully.")

print("\nNumber of clusters:", user_cluster_data["Cluster"].nunique())

print("\nUsers in each cluster:")
print(user_cluster_data["Cluster"].value_counts().sort_index())

print("\nFirst 10 users and their clusters:")
print(user_cluster_data.head(10))

K-Means clustering completed successfully.

Number of clusters: 5

Users in each cluster:
Cluster
0     498
1    1177
2    3411
3     765
4     189
Name: count, dtype: int64

First 10 users and their clusters:
   UserID  Cluster
0       1        2
1       2        1
2       3        2
3       4        2
4       5        2
5       6        2
6       7        2
7       8        1
8       9        2
9      10        0


In [15]:
# Cluster-Based Collaborative Filtering Recommendation Function

def recommend_movies_clustered(
    user_id,
    num_recommendations=10,
    num_similar_users=20
):
    
    # Get the target user's cluster
    target_cluster = user_cluster_data.loc[
        user_cluster_data["UserID"] == user_id,
        "Cluster"
    ].iloc[0]
    
    # Get users belonging to the same cluster
    cluster_users = user_cluster_data[
        user_cluster_data["Cluster"] == target_cluster
    ]["UserID"].values
    
    # Get indices of users in the cluster
    cluster_indices = [
        user_movie_matrix.index.get_loc(uid)
        for uid in cluster_users
    ]
    
    # Get target user's index
    user_index = user_movie_matrix.index.get_loc(user_id)
    
    # Similarity scores within the cluster
    cluster_similarities = user_similarity[
        user_index,
        cluster_indices
    ]
    
    # Remove the target user from possible similar users
    valid_indices = [
        i for i, uid in enumerate(cluster_users)
        if uid != user_id
    ]
    
    cluster_users = cluster_users[valid_indices]
    cluster_similarities = cluster_similarities[valid_indices]
    
    # Select the most similar users
    top_indices = np.argsort(cluster_similarities)[::-1]
    top_indices = top_indices[:num_similar_users]
    
    similar_user_ids = cluster_users[top_indices]
    similar_scores = cluster_similarities[top_indices]
    
    # Convert user IDs to matrix indices
    similar_user_indices = [
        user_movie_matrix.index.get_loc(uid)
        for uid in similar_user_ids
    ]
    
    # Ratings of similar users
    similar_users_ratings = feature_matrix[similar_user_indices]
    
    # Calculate weighted recommendation scores
    weighted_scores = np.dot(
        similar_scores,
        similar_users_ratings
    )
    
    # Exclude movies already rated by target user
    user_ratings = feature_matrix[user_index]
    weighted_scores[user_ratings > 0] = -1
    
    # Get top recommendations
    recommended_indices = np.argsort(weighted_scores)[::-1]
    recommended_indices = recommended_indices[:num_recommendations]
    
    # Movie IDs
    recommended_movie_ids = user_movie_matrix.columns[
        recommended_indices
    ]
    
    # Create recommendation dataframe
    recommendations = movie_features[
        movie_features["MovieID"].isin(recommended_movie_ids)
    ].copy()
    
    # Preserve recommendation order
    recommendation_order = {
        movie_id: rank
        for rank, movie_id in enumerate(recommended_movie_ids)
    }
    
    score_mapping = dict(
        zip(
            recommended_movie_ids,
            weighted_scores[recommended_indices]
        )
    )
    
    recommendations["Recommendation_Rank"] = (
        recommendations["MovieID"].map(recommendation_order)
    )
    
    recommendations["Recommendation_Score"] = (
        recommendations["MovieID"].map(score_mapping)
    )
    
    recommendations = recommendations.sort_values(
        "Recommendation_Rank"
    )
    
    return recommendations.reset_index(drop=True)


print("Cluster-based recommendation function created successfully.")

Cluster-based recommendation function created successfully.


In [16]:
# Test Cluster-Based Recommendations

cluster_recommendations = recommend_movies_clustered(
    user_id=1,
    num_recommendations=10,
    num_similar_users=20
)

print("Cluster-Based Recommendations for User 1:")
print(cluster_recommendations)

Cluster-Based Recommendations for User 1:
   MovieID                                        Movie_Title  \
0     2081                         Little Mermaid, The (1989)   
1      364                              Lion King, The (1994)   
2     2096                             Sleeping Beauty (1959)   
3      596                                   Pinocchio (1940)   
4     2078                            Jungle Book, The (1967)   
5     2085                              101 Dalmatians (1961)   
6     2080                          Lady and the Tramp (1955)   
7     1196  Star Wars: Episode V - The Empire Strikes Back...   
8       34                                        Babe (1995)   
9     2087                                   Peter Pan (1953)   

                                        Genres  Recommendation_Rank  \
0  Animation|Children's|Comedy|Musical|Romance                    0   
1                 Animation|Children's|Musical                    1   
2                 Animation|C

In [17]:
# Step 12: Save Model/Data

import pickle
import os

# Create model directory
os.makedirs("model", exist_ok=True)

# Save cosine similarity matrix
with open("model/user_similarity.pkl", "wb") as file:
    pickle.dump(user_similarity, file)

# Save feature matrix
with open("model/feature_matrix.pkl", "wb") as file:
    pickle.dump(feature_matrix, file)

# Save user-movie matrix
with open("model/user_movie_matrix.pkl", "wb") as file:
    pickle.dump(user_movie_matrix, file)

# Save K-Means model
with open("model/kmeans_model.pkl", "wb") as file:
    pickle.dump(kmeans, file)

# Save user cluster information
with open("model/user_clusters.pkl", "wb") as file:
    pickle.dump(user_cluster_data, file)

# Save movie information
with open("model/movie_features.pkl", "wb") as file:
    pickle.dump(movie_features, file)

print("All model and data files saved successfully.")

print("\nSaved files:")
print(os.listdir("model"))

All model and data files saved successfully.

Saved files:
['feature_matrix.pkl', 'kmeans_model.pkl', 'movie_features.pkl', 'user_clusters.pkl', 'user_movie_matrix.pkl', 'user_similarity.pkl']
